# 02 Data Preparation

## Purpose
Transform raw Statcast data and sprint speed data into a clean event-level dataset for model training.

## Inputs
- `data/raw/statcast_events.parquet`
- `data/raw/sprint_speed.parquet`

## Outputs
- `data/processed/bbe_model_input.parquet`

## Notes
This notebook filters raw events to valid batted-ball events, maps clean outcomes, merges sprint speed, and prepares the dataset used in model training.

In [1]:
import os
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

## Load raw source data

In [2]:
raw_statcast = pd.read_parquet("data/raw/statcast_events.parquet")
ss = pd.read_parquet("data/raw/sprint_speed.parquet")

print("Raw Statcast shape:", raw_statcast.shape)
print("Sprint speed shape:", ss.shape)

Raw Statcast shape: (2143789, 119)
Sprint speed shape: (1728, 11)


## Filter to valid batted-ball events

In [3]:
exclude_events = [
    "catcher_interf",
    "fielders_choice",
    "field_error",
    "fielders_choice_out",
    "sac_fly",
    "sac_fly_double_play",
    "sac_bunt",
    "sac_bunt_double_play"
]

bbe = raw_statcast.copy()
bbe = bbe[bbe["description"] == "hit_into_play"].copy()
bbe = bbe[~bbe["events"].isin(exclude_events)].copy()
bbe = bbe[bbe["game_type"] == "R"].copy()

#CHANGE LATER???
bbe = bbe[
    bbe["launch_speed"].notna() &
    bbe["launch_angle"].notna()
].copy()

print("Filtered BBE shape:", bbe.shape)
bbe.head()

Filtered BBE shape: (361509, 119)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season
0,CH,2023-10-01,89.0,-2.8,5.59,"Robertson, Nick",677008,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,9,"Heston Kjerstad grounds out, first baseman Bob...",R,L,R,BAL,BOS,X,3,ground_ball,2,2,2023,-1.53,0.33,0.333018,2.005061,<NA>,<NA>,<NA>,2,9,Bot,158.28,166.83,<NA>,<NA>,<NA>,<NA>,11.122985,-129.176025,-3.49208,-19.471845,26.055263,-27.922064,3.81,1.74,6,96.4,-17,90.7,1703,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.11,0.147,0.152,0.0,1,0,0,2,73,6,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,-0.001,-0.233,66.5,6.5,0.149,0.233,96.4,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.55,1.53,-1.53,31.7,1.676715,-1.896554,41.830979,30.714944,26.41202,2023
12,FF,2023-10-01,95.8,-2.48,5.88,"Robertson, Nick",663624,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,13,Ryan Mountcastle pops out to second baseman En...,R,R,R,BAL,BOS,X,4,popup,3,1,2023,-0.54,1.58,-1.051937,2.766499,<NA>,<NA>,<NA>,0,9,Bot,168.39,135.05,<NA>,<NA>,<NA>,<NA>,5.071337,-139.246921,-6.233062,-8.158237,33.874278,-10.472415,3.73,1.86,194,68.3,46,97.3,2204,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.13,0.284,0.293,0.0,1,0,0,3,71,5,4-Seam Fastball,1,6,1,6,6,1,1,6,Standard,Standard,212,-0.009,-0.442,69.3,6.4,0.387,0.442,88.0,-5,-5,0.012,0.012,24,26,25,26,1,1,11,1,<NA>,6,0.93,0.54,0.54,42.9,-4.578371,1.280243,40.060882,24.734672,25.749552,2023
22,FC,2023-10-01,83.6,1.42,6.13,"Irvin, Cole",624512,608344,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,13,"Reese McGuire grounds out, pitcher Cole Irvin ...",R,L,L,BAL,BOS,X,1,ground_ball,0,2,2023,-0.06,-0.17,-0.912408,2.294593,<NA>,<NA>,<NA>,1,9,Top,127.46,186.95,<NA>,<NA>,<NA>,<NA>,-5.249558,-121.736232,-2.093667,0.459916,24.107616,-33.636595,3.42,1.7,1,62.6,-64,83.5,2273,6.2,716367,543510,663624,676059,602104,683002,677008,622761,623993,54.27,0.205,0.183,0.0,1,0,0,2,69,4,Cutter,1,6,6,1,6,1,6,1,Standard,Standard,284,0.001,-0.17,61.8,7.6,0.205,0.17,88.0,-5,5,0.007,0.993,29,28,29,28,1,1,19,1,<NA>,<NA>,3.43,-0.06,-0.06,39.2,0.237634,20.464121,24.866288,47.255717,20.158185,2023
26,SI,2023-10-01,89.3,1.14,6.33,"Irvin, Cole",622569,608344,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,1,Pablo Reyes grounds out to first baseman Ryan ...,R,R,L,BAL,BOS,X,3,ground_ball,0,2,2023,1.28,0.96,-0.297831,3.19833,<NA>,<NA>,<NA>,0,9,Top,153.72,172.92,<NA>,<NA>,<NA>,<NA>,-6.249503,-129.945518,-3.724564,15.822763,27.24765,-20.677397,3.36

## Create clean outcome labels

In [4]:
def map_outcome(event):
    if event == "single":
        return "1B"
    elif event == "double":
        return "2B"
    elif event == "triple":
        return "3B"
    elif event == "home_run":
        return "HR"
    else:
        return "OUT"

bbe["outcome_label"] = bbe["events"].apply(map_outcome)

print(bbe["outcome_label"].value_counts(dropna=False))

outcome_label
OUT    240994
1B      77802
2B      23721
HR      16959
3B       2033
Name: count, dtype: int64


## Create event identifier column

In [5]:
bbe = bbe.reset_index(drop=True)

bbe["bbe_id"] = (
    bbe["game_date"].astype(str) + "_" +
    bbe["batter"].astype(str) + "_" +
    bbe["pitcher"].astype(str) + "_" +
    bbe.index.astype(str)
)

bbe[["bbe_id", "game_date", "batter", "pitcher", "events"]].head()

,bbe_id,game_date,batter,pitcher,events
0,2023-10-01_677008_687798_0,2023-10-01,677008,687798,field_out
1,2023-10-01_663624_687798_1,2023-10-01,663624,687798,field_out
2,2023-10-01_624512_608344_2,2023-10-01,624512,608344,field_out
3,2023-10-01_622569_608344_3,2023-10-01,622569,608344,field_out
4,2023-10-01_623993_670167_4,2023-10-01,623993,670167,field_out


## Prepare sprint speed data

In [6]:
ss_clean = ss.copy()
ss_clean = ss_clean.rename(columns={
    "last_name, first_name": "player_name"
})
ss_clean = ss_clean[[
    "season",
    "player_id",
    "player_name",
    "sprint_speed"
]].copy()

ss_clean["season"] = pd.to_numeric(ss_clean["season"], errors="coerce")
ss_clean["player_id"] = pd.to_numeric(ss_clean["player_id"], errors="coerce")
ss_clean["sprint_speed"] = pd.to_numeric(ss_clean["sprint_speed"], errors="coerce")

print("Clean sprint speed shape:", ss_clean.shape)
ss_clean.head()

Clean sprint speed shape: (1728, 4)


,season,player_id,player_name,sprint_speed
0,2023,682829,"De La Cruz, Elly",30.5
1,2023,677951,"Witt Jr., Bobby",30.5
2,2023,680118,"Blanco, Dairon",30.3
3,2023,607208,"Turner, Trea",30.3
4,2023,669352,"Thompson, Bubba",30.2


## Merge sprint speed onto batted-ball events

In [7]:
bbe["season"] = pd.to_numeric(bbe["season"], errors="coerce")
bbe["batter"] = pd.to_numeric(bbe["batter"], errors="coerce")

bbe = bbe.merge(
    ss_clean[["season", "player_id", "player_name", "sprint_speed"]],
    left_on=["season", "batter"],
    right_on=["season", "player_id"],
    how="left"
)

bbe = bbe.drop(columns=["player_id"], errors="ignore")

print("Shape after sprint speed merge:", bbe.shape)
print("Missing sprint speed rows:", bbe["sprint_speed"].isna().sum())

Shape after sprint speed merge: (361509, 123)
Missing sprint speed rows: 1988


In [ ]:
#CHANGE LATER???
season_medians = bbe.groupby("season")["sprint_speed"].transform("median")
bbe["sprint_speed"] = bbe["sprint_speed"].fillna(season_medians)
print("Missing sprint speed after fill:", bbe["sprint_speed"].isna().sum())

Missing sprint speed after fill: 0


## Select modeling columns

In [9]:
model_input = bbe.rename(columns={
    "batter": "batter_id",
    "player_name": "batter_name"
}).copy()

model_cols = [
    "bbe_id",
    "game_date",
    "season",
    "batter_id",
    "batter_name",
    "pitcher",
    "events",
    "outcome_label",
    "actual_result_value",
    "launch_speed",
    "launch_angle",
    "sprint_speed",
    "stand",
    "p_throws",
    "balls",
    "strikes",
    "pitch_type",
    "zone",
    "bb_type",
    "hc_x",
    "hc_y",
    "home_team",
    "away_team"
]

model_input = model_input[[c for c in model_cols if c in model_input.columns]].copy()

print("Final model input shape:", model_input.shape)
model_input.head()

Final model input shape: (361509, 21)


,bbe_id,game_date,season,batter_id,pitcher,events,outcome_label,launch_speed,launch_angle,sprint_speed,stand,p_throws,balls,strikes,pitch_type,zone,bb_type,hc_x,hc_y,home_team,away_team
0,2023-10-01_677008_687798_0,2023-10-01,2023,677008,687798,field_out,OUT,96.4,-17,27.4,L,R,2,2,CH,9,ground_ball,158.28,166.83,BAL,BOS
1,2023-10-01_663624_687798_1,2023-10-01,2023,663624,687798,field_out,OUT,68.3,46,28.2,R,R,3,1,FF,13,popup,168.39,135.05,BAL,BOS
2,2023-10-01_624512_608344_2,2023-10-01,2023,624512,608344,field_out,OUT,62.6,-64,25.9,L,L,0,2,FC,13,ground_ball,127.46,186.95,BAL,BOS
3,2023-10-01_622569_608344_3,2023-10-01,2023,622569,608344,field_out,OUT,80.5,-38,27.9,R,L,0,2,SI,1,ground_ball,153.72,172.92,BAL,BOS
4,2023-10-01_623993_670167_4,2023-10-01,2023,623993,670167,field_out,OUT,82.0,43,26.7,L,R,2,0,ST,5,fly_ball,59.68,124.3,BAL,BOS


## Quick data validation

In [10]:
print(model_input.isna().sum().sort_values(ascending=False).head(15))
print()
print(model_input["outcome_label"].value_counts())
print()
print(model_input[["launch_speed", "launch_angle", "sprint_speed"]].describe())

hc_x             145
hc_y             145
season             0
game_date          0
bbe_id             0
pitcher            0
batter_id          0
events             0
outcome_label      0
sprint_speed       0
stand              0
launch_speed       0
launch_angle       0
balls              0
p_throws           0
dtype: int64

outcome_label
OUT    240994
1B      77802
2B      23721
HR      16959
3B       2033
Name: count, dtype: int64

       launch_speed  launch_angle   sprint_speed
count      361509.0      361509.0  361509.000000
mean      88.786265     13.253653      27.331405
std       14.551882      28.58139       1.325386
min             5.6         -90.0      22.700000
25%            80.3          -4.0      26.500000
50%            91.6          14.0      27.400000
75%            99.7          31.0      28.300000
max           122.9          90.0      30.500000


## Save processed dataset

In [11]:
os.makedirs("data/processed", exist_ok=True)
model_input.to_parquet("data/processed/bbe_model_input.parquet", index=False)

print("Saved: data/processed/bbe_model_input.parquet")

Saved: data/processed/bbe_model_input.parquet
